<a href="https://colab.research.google.com/github/mandar-solanki/exercises/blob/main/2604-Applied-Data-Modelling-using-Gradio/0501%20Spam%20Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Gradio Based Spam Classifier

#### 1) Load Data

In [1]:
filepath = r'https://raw.githubusercontent.com/suyashi29/python-su/refs/heads/master/Applied%20Data%20Modelling%20using%20Gradio/spam_data.csv'


In [2]:
import pandas as pd


In [11]:
df = pd.read_csv(filepath)

print(df.head())
print()
print(df.isnull().sum())
print()
print(df.describe())


                                     text label
0                      See you soon Noted   ham
1          Project discussion today Noted   ham
2                 Earn cash fast Call now  spam
3        Get loan instantly Limited offer  spam
4  Exclusive deal just for you Click here  spam

text     0
label    0
dtype: int64

                                       text label
count                                  1200  1200
unique                                  145     2
top     Congratulations! You won Click here   ham
freq                                     16   600


#### 2) Data Preprocessing

In [4]:
import re


In [13]:
def clean_text(s):
    return re.sub(r'[^a-zA-Z0-9\s]','',s.lower())

def create_y(s):
    if s=='spam':
        return 1
    return 0



In [14]:
df['X'] = df['text'].apply(clean_text)
df['y'] = df['label'].apply(create_y)
print(df.head())


                                     text label  \
0                      See you soon Noted   ham   
1          Project discussion today Noted   ham   
2                 Earn cash fast Call now  spam   
3        Get loan instantly Limited offer  spam   
4  Exclusive deal just for you Click here  spam   

                                        X  y  
0                      see you soon noted  0  
1          project discussion today noted  0  
2                 earn cash fast call now  1  
3        get loan instantly limited offer  1  
4  exclusive deal just for you click here  1  


In [16]:
# Create X,y
X = df['X'].to_list()
y = df['y'].to_list()


#### 3) Feature Engineering

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


In [18]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(X)


In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)


In [24]:
#X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)


#### 4) Model Training & Evaluation
Naive Bayes & Logistic Regression

In [23]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report
from sklearn.pipeline import Pipeline


In [39]:
# Model 1 - Naive Bayes
model_nb = Pipeline([
    ('preprocessor', tfidf_vectorizer),
    ('regressor',MultinomialNB())
])

model_nb.fit(X_train, y_train)

y_pred_nb = model_nb.predict(X_test)

accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
cnf_matrix_nb = confusion_matrix(y_test, y_pred_nb)

def model_evaluation_nb():
    result = f'Accuracy : {accuracy_nb:.3f}\n'
    result += f'Precision : {precision_nb:.3f}\n'
    result += f'Recall : {recall_nb:.3f}\n'
    result += f'Confusion Matrix :\n{cnf_matrix_nb}'
    return result


In [40]:
model_evaluation_nb()

'Accuracy : 1.000\nPrecision : 1.000\nRecall : 1.000\nConfusion Matrix :\n[[111   0]\n [  0 129]]'

In [41]:
# Model 2 - Logistic Regression
model_lr = Pipeline([
    ('preprocessor', tfidf_vectorizer),
    ('regressor',LogisticRegression())
])

model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
cnf_matrix_lr = confusion_matrix(y_test, y_pred_lr)

def model_evaluation_lr():
    result = f'Accuracy : {accuracy_lr:.3f}\n'
    result += f'Precision : {precision_lr:.3f}\n'
    result += f'Recall : {recall_lr:.3f}\n'
    result += f'Confusion Matrix :\n{cnf_matrix_lr}'
    return result


In [42]:
model_evaluation_lr()

'Accuracy : 1.000\nPrecision : 1.000\nRecall : 1.000\nConfusion Matrix :\n[[111   0]\n [  0 129]]'

In [51]:
# Predict functions

def predict_nb(s):
    clean = [clean_text(s)]
    pred = model_nb.predict(clean)[0]
    if pred == 1:
        return 'Spam'
    return 'Not Spam'

def predict_lr(s):
    clean = [clean_text(s)]
    pred = model_lr.predict(clean)[0]
    if pred == 1:
        return 'Spam'
    return 'Not Spam'



#### 5) Gradio App

In [46]:
import gradio as gr

In [52]:
# Gradio UI

with gr.Blocks(title='Spam Classifier') as app:
    gr.Markdown('## Spam Prediction Models')

    # EDA
    with gr.Tab('EDA'):
        gr.Markdown('### Dataset Overview')
        gr.Dataframe(df.head())
        gr.Markdown('### No nulls observed')
        gr.TextArea(df.isnull().sum())
        gr.Markdown('### Summary Statistics')
        gr.TextArea(df.describe())

    # Model Performance Comparison
    with gr.Tab('Model Comparison'):
        gr.Markdown('Naive Bayes and Logistic Regression models were trained. Below is their performance on the test data.')
        with gr.Row():
            with gr.Column():
                gr.Markdown('### Naive Bayes Regression Metrics')
                but_model_metrics_nb = gr.Button('Show Model Metrics')
                out_model_metrics_nb = gr.TextArea()
                but_model_metrics_nb.click(fn=model_evaluation_nb, outputs=out_model_metrics_nb)
            with gr.Column():
                gr.Markdown('### Logistic Regression Metrics')
                but_model_metrics_lr = gr.Button('Show Model Metrics')
                out_model_metrics_lr = gr.TextArea()
                but_model_metrics_lr.click(fn=model_evaluation_lr, outputs=out_model_metrics_lr)

    # Prediction
    with gr.Tab('Model Prediction'):
        gr.Markdown('### Predict Hiring')
        input_text = gr.Textbox(label='Enter the text to check for Spam : ')
        but_pred = gr.Button('Predict Spam or Not Spam')
        with gr.Row():
            with gr.Column():
                gr.Markdown('#### Naive Bayes says : ')
                out_pred_nb = gr.Textbox()
                but_pred.click(
                    fn=predict_nb,
                    inputs=[input_text],
                    outputs=out_pred_nb
                )
            with gr.Column():
                gr.Markdown('#### Logistic Regression says : ')
                out_pred_lr = gr.Textbox()
                but_pred.click(
                    fn=predict_lr,
                    inputs=[input_text],
                    outputs=out_pred_lr
                )



In [57]:
app.launch(share=False, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.


In [56]:
app.close()


Closing server running on port: 7860
